# Beam Dataset Generation for Language Model Training

This notebook generates synthetic beam mechanics datasets for training language models on structural engineering problems. It creates beam configurations with varying parameters, solves them symbolically, and generates question-answer pairs using LLMs.

## Overview

The dataset generation process involves:
1. Creating beam configurations with symbolic parameters (lengths, loads, supports)
2. Solving beam equations symbolically to obtain reactions, moments, and deflections
3. Generating natural language questions using LLMs
4. Extracting ground-truth answers from the solved beam equations
5. Formatting everything into a HuggingFace-compatible dataset

## Workflow

1. **Setup**: Import dependencies and configure symbeam path
2. **Configuration**: Set beam parameters (lengths, loads, positions) and output options
3. **Generation**: Create beam configurations and solve symbolically
4. **Conversion**: Transform to HuggingFace Dataset format
5. **QA Generation**: Generate questions using LLM (optional)
6. **Post-processing**: Extract answers and rename columns
7. **Output**: Save intermediary datasets locally, upload final dataset to Hub

## Output Structure

- **Intermediary datasets** are saved to `intermediary_datasets/` subdirectory:
  - `01_initial_dataset.json` - Initial beam analysis dataset with symbolic solutions
  - `02_dataset_with_qa.json` - Dataset with LLM-generated questions
  - `03_final_dataset.json` - Final processed dataset with cleaned queries and answers
- **Final dataset** is uploaded to HuggingFace Hub for public access

## Requirements

- Python 3.8+
- symbeam library (for symbolic beam analysis)
- HuggingFace datasets library
- vLLM (for LLM inference)

## Features

This notebook supports:
- **Multiple loads** at explicit positions
- **Custom support positions** along the beam
- **Symbolic beam analysis** with SymPy
- **LLM-based question generation** (optional)
- **HuggingFace dataset** format for easy sharing


### Setup

Import dependencies and configure symbeam path.

In [1]:
# Standard library imports
import sys
import os
import json

# Scientific computing libraries
import numpy as np  # For numerical operations and array handling
import sympy  # For symbolic mathematics
from sympy import Basic  # Base class for SymPy objects

# HuggingFace datasets library for dataset creation and management
from datasets import Dataset


In [2]:
## 2. Configure symbeam Library Path

# Determine the current directory (works in both script and notebook contexts)
current_dir = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
symbeam_path = os.path.join(current_dir, "symbeam_v2")

# Add symbeam_v2 to Python path if it exists, otherwise raise an error
if os.path.exists(symbeam_path):
    sys.path.insert(0, os.path.abspath(symbeam_path))
else:
    raise FileNotFoundError(f"symbeam_v2 directory not found at {symbeam_path}")

# Import symbeam library for symbolic beam analysis
from symbeam import beam

# Import symbolic variables for beam parameters
# L = length, E = Young's modulus, I = moment of inertia
# P = point load, M = moment, q = distributed load, x = position along beam
from sympy.abc import L, E, I, P, M, q, x

### Load Configuration Guide

In [3]:
# ============================================================================
# CORE FUNCTIONS FOR BEAM DATASET GENERATION
# ============================================================================

def create_beam_configuration(P_val=-P, q_val=0, M_val=0, L_val=L, E_val=E, I_val=I, L_loc=0.5, load_positions=None, load_values=None, support_positions=None):
    """
    Create a beam with specified parameters and solve symbolically.
    
    This function creates a simply supported beam (pin and roller supports),
    applies loads, and solves for reactions, moments, and deflections.
    
    Args:
        P_val: Point load value (default: -P, negative indicates downward) - used if load_positions is None
        q_val: Distributed load value (default: 0)
        M_val: Point moment value (default: 0)
        L_val: Beam length (default: L, symbolic)
        E_val: Young's modulus (default: E, symbolic)
        I_val: Moment of inertia (default: I, symbolic)
        L_loc: Normalized load position (0.0 to 1.0, default: 0.5) - used if load_positions is None
        load_positions: List of normalized load positions (0.0 to 1.0) - if provided, overrides L_loc
        load_values: List of load values corresponding to load_positions - if provided, overrides P_val
        support_positions: List [pin_pos, roller_pos] normalized positions (0.0 to 1.0) or None for default [0.0, 1.0]
    
    Returns:
        tuple: (plot_data, subs_dict, support_info)
            - plot_data: Dictionary containing solved beam data (reactions, moments, deflections)
            - subs_dict: Dictionary of symbolic substitutions used in solving
            - support_info: Dictionary with support positions {'pin_pos': pin_pos, 'roller_pos': roller_pos}
    """
    # Create a new beam instance with specified length
    new_beam = beam(L_val)
    
    # Set material properties (constant along the beam)
    new_beam.set_young(0, L_val, E_val)  # Young's modulus
    new_beam.set_inertia(0, L_val, I_val)  # Moment of inertia
    
    # Add supports: pin and roller at specified positions (default: pin at x=0, roller at x=L)
    if support_positions is None:
        pin_pos = 0.0
        roller_pos = 1.0
    else:
        if len(support_positions) != 2:
            raise ValueError(f"support_positions must be a list of 2 values [pin_pos, roller_pos], got {support_positions}")
        # Convert to float to avoid SymPy boolean evaluation issues
        pin_pos = support_positions[0]
        roller_pos = support_positions[1]
        
        if pin_pos < 0 or pin_pos > 1 or roller_pos < 0 or roller_pos > 1:
            raise ValueError(f"Support positions must be between 0 and 1, got pin={pin_pos}, roller={roller_pos}")
    
    # Convert normalized positions to actual coordinates
    pin_pos_val = pin_pos * L_val
    roller_pos_val = roller_pos * L_val
    
    new_beam.add_support(pin_pos_val, 'pin')
    new_beam.add_support(roller_pos_val, 'roller')
    
    # Handle multiple loads if provided
    if load_positions is not None and load_values is not None:
        if len(load_positions) != len(load_values):
            raise ValueError(f"load_positions and load_values must have the same length, got {len(load_positions)} and {len(load_values)}")
        
        # Add multiple point loads
        for pos, load_val in zip(load_positions, load_values):
            # Convert to float to avoid SymPy boolean evaluation issues
            pos_float = pos
            if pos_float < 0 or pos_float > 1:
                raise ValueError(f"Load position must be between 0 and 1, got {pos}")
            L_loc_val = pos_float * L_val
            new_beam.add_point_load(L_loc_val, load_val)
    else:
        # Single load mode (backward compatibility)
        # Validate load position (must be between 0 and 1)
        # Convert to float to avoid SymPy boolean evaluation issues
        L_loc_float = L_loc
        if L_loc_float < 0 or L_loc_float > 1:
            raise ValueError(f"L_loc must be between and including 0 and 1, got {L_loc}")
        # Convert normalized position to actual coordinate
        L_loc_val = L_loc_float * L_val
        # Add point load at specified location
        new_beam.add_point_load(L_loc_val, P_val)
    
    # Solve the beam symbolically
    # subs_dict contains the symbolic parameters for substitution
    subs_dict = {'P': P_val, 'q': q_val, 'L': L_val, 'M': M_val, 'E': E_val, 'I': I_val}
    plot_data = new_beam.solve_v3(subs=subs_dict)
    
    # Store support information
    support_info = {'pin_pos': pin_pos, 'roller_pos': roller_pos}
    
    return plot_data, subs_dict, support_info


### Configuration

Set beam parameters (lengths, loads, positions) and output options.

In [4]:
# ============================================================================
# Function: Generate beam dataset
# ============================================================================

def generate_beam_dataset(lengths=None, loads=None, n_positions=21, q_val=0, M_val=0, E_val=E, I_val=I, load_configs=None, parse_value_func=None, support_positions=None):
    """
    Generate beam configurations for all combinations of parameters.
    
    This function creates all possible combinations of beam lengths, loads, and load positions,
    solves each configuration symbolically, and collects the results.
    
    Args:
        lengths: List of beam lengths (default: [L, 2*L])
        loads: List of point loads (default: [-P, -2*P]) - used only if load_configs is None
        n_positions: Number of load positions along beam (default: 21) - used only if load_configs is None
        q_val: Distributed load value (default: 0)
        M_val: Point moment value (default: 0)
        E_val: Young's modulus (default: E)
        I_val: Moment of inertia (default: I)
        load_configs: List of dicts with 'load' and 'positions' keys - if provided, overrides loads/n_positions
        parse_value_func: Function to parse string values to SymPy expressions (optional)
        support_positions: List [pin_pos, roller_pos] normalized positions (0.0 to 1.0) or None for default [0.0, 1.0]
    
    Returns:
        dict: Dictionary containing list of beam configurations with their solved data
    """
    # Set default values if not provided
    if lengths is None:
        lengths = [L, 2*L]
    if loads is None:
        loads = [-P, -2*P]
    
    # Parse values if parse function provided
    if parse_value_func:
        lengths = [parse_value_func(l) for l in lengths]
        loads = [parse_value_func(p) for p in loads]
        q_val = parse_value_func(q_val)
        M_val = parse_value_func(M_val)
        E_val = parse_value_func(E_val)
        I_val = parse_value_func(I_val)
    
    # Initialize data structure to store all configurations
    all_beam_data = {'beam_configurations': []}
    config_id = 0
    
    # Handle load_configs format (new format with explicit positions)
    if load_configs is not None:
        # Parse load_configs
        parsed_load_configs = []
        for config in load_configs:
            load_val = parse_value_func(config['load']) if parse_value_func else config['load']
            positions = config['positions']  # Already normalized 0.0-1.0
            parsed_load_configs.append({'load': load_val, 'positions': positions, 'support_positions': config.get('support_positions')})
        
        # Generate configurations: length × load_config
        total_configs = len(lengths) * len(parsed_load_configs)
        print(f"Generating {total_configs} beam configurations with explicit load positions...")
        
        for L_val in lengths:
            for load_config in parsed_load_configs:
                try:
                    load_val = load_config['load']
                    positions = load_config['positions']
                    
                    # Ensure positions are floats (not SymPy expressions)
                    positions_float = [pos for pos in positions]
                    
                    # Determine support positions (config-specific overrides global default)
                    config_support_positions = load_config.get('support_positions')
                    support_pos_float = None
                    if config_support_positions is not None:
                        support_pos_float = [pos for pos in config_support_positions]
                    elif support_positions is not None:
                        support_pos_float = [pos for pos in support_positions]
                    
                    # Progress indicator
                    print(f"Processing config {config_id+1}/{total_configs}: L={L_val}, Load={load_val}, Positions={positions_float}")
                    
                    # Create and solve beam configuration with multiple loads
                    plot_data, subs_dict, support_info = create_beam_configuration(
                        L_val=L_val,
                        load_positions=positions_float,  # Use float-converted positions
                        load_values=[load_val] * len(positions_float),  # Same load value at all positions
                        q_val=q_val, M_val=M_val, E_val=E_val, I_val=I_val,
                        support_positions=support_pos_float  # Use float-converted support positions
                    )
                    
                    # Convert SymPy objects to JSON-serializable format
                    plot_data_serializable = make_json_serializable(plot_data)
                    subs_dict_serializable = make_json_serializable(subs_dict)
                    
                    # Store configuration data
                    config_entry = {
                        'configuration_id': config_id,
                        'load_positions': positions_float,  # List of all load positions (as floats)
                        'load_values': [str(load_val)] * len(positions_float),  # List of load values
                        'support_positions': [float(support_info['pin_pos']), float(support_info['roller_pos'])],  # Support positions (as floats)
                        'parameters': subs_dict_serializable,
                        'plot_data': plot_data_serializable
                    }
                    all_beam_data['beam_configurations'].append(config_entry)
                    config_id += 1
                    
                except Exception as e:
                    # Skip configurations that fail (e.g., loads at support positions, invalid parameters)
                    import traceback
                    error_type = type(e).__name__
                    error_msg = str(e)
                    print(f"\n{'='*80}")
                    print(f"ERROR processing config {config_id+1}/{total_configs}")
                    print(f"{'='*80}")
                    print(f"Configuration parameters:")
                    print(f"  - Beam length (L_val): {L_val}")
                    print(f"  - Load value: {load_val}")
                    print(f"  - Load positions (original): {positions}")
                    print(f"  - Load positions (converted to float): {positions_float}")
                    support_desc = support_pos_float if support_pos_float is not None else 'Default [0.0, 1.0]'
                    print(f"  - Support positions: {support_desc}")
                    print(f"  - q_val: {q_val}, M_val: {M_val}, E_val: {E_val}, I_val: {I_val}")
                    print(f"\nError details:")
                    print(f"  - Error type: {error_type}")
                    print(f"  - Error message: {error_msg}")
                    print(f"\nFull traceback:")
                    traceback.print_exc()
                    print(f"{'='*80}\n")
                    continue
    else:
        # Legacy mode: evenly spaced positions
        # Generate evenly spaced load positions from 0.0 to 1.0
        L_loc_values = np.linspace(0, 1, n_positions)
        
        # Calculate total number of configurations to generate
        total_configs = len(lengths) * len(loads) * n_positions
        print(f"Generating {total_configs} beam configurations...")
        
        # Generate all combinations: length × load × position
        for L_val in lengths:
            for P_val in loads:
                for L_loc in L_loc_values:
                    try:
                        # Ensure L_loc is a float (not SymPy expression)
                        L_loc_float = L_loc
                        
                            # Determine support positions (config-specific overrides global default)
                        support_pos_float = None
                        if support_positions is not None:
                            support_pos_float = [pos for pos in support_positions]
                        
                        # Progress indicator
                        print(f"Processing config {config_id+1}/{total_configs}: L={L_val}, P={P_val}, Load pos={L_loc_float:.3f}")
                        
                        # Create and solve beam configuration
                        plot_data, subs_dict, support_info = create_beam_configuration(
                            L_val=L_val, P_val=P_val, L_loc=L_loc_float,  # Use float-converted L_loc
                            q_val=q_val, M_val=M_val, E_val=E_val, I_val=I_val,
                            support_positions=support_pos_float  # Use float-converted support positions
                        )
                        
                        # Convert SymPy objects to JSON-serializable format
                        plot_data_serializable = make_json_serializable(plot_data)
                        subs_dict_serializable = make_json_serializable(subs_dict)
                        
                        # Store configuration data
                        config_entry = {
                            'configuration_id': config_id,
                            'load_positions': [L_loc_float],  # List for potential multiple loads (as float)
                            'load_values': [str(P_val)],  # List of load values
                            'support_positions': [float(support_info['pin_pos']), float(support_info['roller_pos'])],  # Support positions (as floats)
                            'parameters': subs_dict_serializable,
                            'plot_data': plot_data_serializable
                        }
                        all_beam_data['beam_configurations'].append(config_entry)
                        config_id += 1
                        
                    except Exception as e:
                        # Skip configurations that fail (e.g., loads at support positions, invalid parameters)
                        import traceback
                        error_type = type(e).__name__
                        error_msg = str(e)
                        print(f"\n{'='*80}")
                        print(f"ERROR processing config {config_id+1}/{total_configs}")
                        print(f"{'='*80}")
                        print(f"Configuration parameters:")
                        print(f"  - Beam length (L_val): {L_val}")
                        print(f"  - Point load value (P_val): {P_val}")
                        print(f"  - Load position (L_loc, original): {L_loc}")
                        print(f"  - Load position (L_loc_float, converted): {L_loc_float}")
                        support_desc = support_pos_float if support_pos_float is not None else 'Default [0.0, 1.0]'
                        print(f"  - Support positions: {support_desc}")
                        print(f"  - q_val: {q_val}, M_val: {M_val}, E_val: {E_val}, I_val: {I_val}")
                        print(f"\nError details:")
                        print(f"  - Error type: {error_type}")
                        print(f"  - Error message: {error_msg}")
                        print(f"\nFull traceback:")
                        traceback.print_exc()
                        print(f"{'='*80}\n")
                        continue
    
    print(f"\nSuccessfully processed {len(all_beam_data['beam_configurations'])} configurations")
    return all_beam_data

In [5]:
# ============================================================================
# Function: Create HuggingFace dataset
# ============================================================================

def create_huggingface_dataset(
    lengths=None,
    loads=None,
    n_positions=21,
    q_val=0,
    M_val=0,
    E_val="E",
    I_val="I",
    load_configs=None,
    support_positions=None
):
    """Generate the beam dataset for all symbolic lengths and loads and prepare it for Hugging Face Hub
    
    This version supports the new load_configs format and support_positions.
    """
    print("Generating beam dataset...")
    
    # Parse values using parse_value function
    lengths = [parse_value(l) for l in (lengths or [L, 2*L])]
    loads = [parse_value(p) for p in (loads or [-P, -2*P])]
    q_val = parse_value(q_val)
    M_val = parse_value(M_val)
    E_val = parse_value(E_val)
    I_val = parse_value(I_val)

    # Generate beam dataset
    beam_data = generate_beam_dataset(
        lengths=lengths,
        loads=loads,
        n_positions=n_positions,
        q_val=q_val,
        M_val=M_val,
        E_val=E_val,
        I_val=I_val,
        load_configs=load_configs,
        parse_value_func=parse_value,
        support_positions=support_positions
    )
    
    configurations = beam_data['beam_configurations']
    dataset_data = []
    for config in configurations:
        plot_data = config['plot_data']
        
        # Handle both new format (load_positions list) and legacy format (single load_position)
        load_positions = config.get('load_positions', [config.get('load_position', 0.5)])
        load_values = config.get('load_values', [str(config.get('parameters', {}).get('P', '-P'))])
        support_pos = config.get('support_positions', [0.0, 1.0])  # Default if not present
        
        row = {
            'configuration_id': config['configuration_id'],
            'load_position': load_positions[0] if len(load_positions) == 1 else None,  # Legacy compatibility
            'load_positions': load_positions,  # New format
            'load_values': load_values,  # New format
            'support_positions': support_pos,  # New format: [pin_pos, roller_pos]
            'parameters': json.dumps(config['parameters']),
            'x_coordinates': plot_data['x_coord'],
            'shear_force': plot_data['shear_force'],
            'bending_moment': plot_data['bending_moment'],
            'slope': plot_data['slope'],
            'deflection': plot_data['deflection'],
            'shear_force_info': json.dumps(plot_data['shear_force_info']),
            'bending_moment_info': json.dumps(plot_data['bending_moment_info']),
            'slope_info': json.dumps(plot_data['slope_info']),
            'deflection_info': json.dumps(plot_data['deflection_info']),
            'points': json.dumps(plot_data['points']),
            'segments': json.dumps(plot_data['segments']),
            'reactions': json.dumps(plot_data['reactions']),
            'internal_loads': json.dumps(plot_data['internal_loads']),
            'deflections': json.dumps(plot_data['deflections'])
        }
        dataset_data.append(row)
    
    dataset = Dataset.from_list(dataset_data)
    dataset.info.description = """
    Beam analysis dataset with varying symbolic lengths and loads.
    Each row contains:
    - configuration_id: Unique identifier for the configuration
    - load_position: Normalized position of the point load (0.0 to 1.0) - legacy format
    - load_positions: List of normalized positions for multiple loads - new format
    - load_values: List of load values corresponding to load_positions - new format
    - support_positions: List [pin_pos, roller_pos] normalized positions (0.0 to 1.0) - new format
    - parameters: JSON string of all beam parameters (symbolic)
    - x_coordinates: List of x-coordinates for plotting
    - shear_force: List of shear force values
    - bending_moment: List of bending moment values
    - slope: List of slope/rotation values
    - deflection: List of deflection values
    - *_info: JSON strings containing local extrema and null points for each quantity
    """
    dataset.info.license = "MIT"
    dataset.info.homepage = "https://github.com/your-repo/beam-analysis"
    return dataset

In [6]:
# ============================================================================
# Define parse_value function if not already available
# ============================================================================
# This function is needed by create_huggingface_dataset
# It may already be defined in a later cell, but we define it here for safety

try:
    # Check if parse_value is already defined
    _ = parse_value
    print("parse_value already defined, using existing definition")
except NameError:
    def parse_value(val):
        """
        Parse a value that may be a string, number, or SymPy expression.
        
        This function handles the conversion of configuration parameters that may be
        provided as strings (e.g., "2*L") into SymPy expressions.
        
        Args:
            val: Value to parse (int, float, string, or SymPy expression)
        
        Returns:
            Parsed value (number or SymPy expression)
        """
        # If already a number, return as-is
        if isinstance(val, (int, float)):
            return val
        
        # Try to parse as a float
        try:
            return float(val)
        except Exception:
            # Try to parse as a SymPy expression (e.g., "2*L" -> 2*L)
            try:
                return sympy.sympify(val, locals={"L": L, "P": P, "E": E, "I": I, "M": M, "q": q})
            except Exception:
                # Return original value if parsing fails
                return val
    print("parse_value defined locally")

def make_json_serializable(obj):
    """
    Recursively convert SymPy objects to JSON-serializable types.
    
    SymPy symbolic expressions cannot be directly serialized to JSON.
    This function converts them to strings or floats where possible.
    
    Args:
        obj: Object that may contain SymPy Basic objects
    
    Returns:
        JSON-serializable version of the object
    """
    # If it's a SymPy Basic object (symbol, expression, etc.)
    if isinstance(obj, Basic):
        # Try to evaluate to a float if it's a numeric expression
        try:
            return float(obj)
        except Exception:
            # Otherwise convert to string representation
            return str(obj)
    # Recursively process dictionaries
    elif isinstance(obj, dict):
        return {make_json_serializable(k): make_json_serializable(v) for k, v in obj.items()}
    # Recursively process lists, tuples, and sets
    elif isinstance(obj, (list, tuple, set)):
        return [make_json_serializable(i) for i in obj]
    # Return primitive types as-is
    else:
        return obj

parse_value defined locally


### Generation

Create beam configurations and solve symbolically.

In [7]:
# ============================================================================
# OUTPUT CONFIGURATION
# ============================================================================
# Base filename for the final dataset
output_filename = "BeamRL-EvalData.json"

# Directory where intermediary datasets will be saved
# This directory will be created automatically if it doesn't exist
intermediary_dir = "intermediary_datasets"

# ============================================================================
# HUGGINGFACE HUB CONFIGURATION
# ============================================================================
# Repository name on HuggingFace Hub (format: username/dataset-name)
repo_name = f"tphage/{output_filename.replace('.json', '')}"

# Set to True if you want the dataset to be private
private = False

# HuggingFace token (set to None to use cached credentials from huggingface-cli login)
hf_token = None

In [8]:
# ============================================================================
# BEAM PARAMETERS
# ============================================================================
# These parameters define the beam configurations to generate.
# The script will create all combinations of lengths × loads × positions.

lengths = ["9*L"]
loads = ["-11*P"]

# Number of load positions along the beam (normalized 0.0 to 1.0)
# Loads will be placed at evenly spaced positions from start to end
n_positions = 21

# Distributed load value (currently not used, set to 0)
q_val = 0

# Point moment value (currently not used, set to 0)
M_val = 0

# Material properties (kept symbolic for general solutions)
E_val = "E"  # Young's modulus
I_val = "I"  # Moment of inertia

# Generate the HuggingFace dataset using configured parameters
load_configs = [
    {'load': "-13*P", 'positions': [0.3333]},
    {'load': "-13*P", 'positions': [0.6667]},
    {'load': "-13*P", 'positions': [0.125]},
    {'load': "-13*P", 'positions': [0.525]},
    
    {'load': "-13*P", 'positions': [0.3333, 0.6667]},
    {'load': "-13*P", 'positions': [0.125, 0.525]},
    {'load': "-13*P", 'positions': [0.3333, 0.125]},
    {'load': "-13*P", 'positions': [0.667, 0.525]},

    {'load': "-13*P", 'positions': [0.3333, 0.5, 0.6667]},
    {'load': "-13*P", 'positions': [0.125, 0.5, 0.875]},
    {'load': "-13*P", 'positions': [0.3333, 0.525, 0.6667]},
    {'load': "-13*P", 'positions': [0.125, 0.525, 0.667]},

    {'load': "-13*P", 'positions': [0.45], 'support_positions': [0.0, 0.9]},
    {'load': "-13*P", 'positions': [1.0], 'support_positions': [0.0, 0.9]},
    {'load': "-13*P", 'positions': [0.45, 1.0], 'support_positions': [0.0, 0.9]},

    {'load': "-13*P", 'positions': [0.55], 'support_positions': [0.1, 1.0]},
    {'load': "-13*P", 'positions': [0.0], 'support_positions': [0.1, 1.0]},
    {'load': "-13*P", 'positions': [0.55, 0.0], 'support_positions': [0.1, 1.0]},

    {'load': "-13*P", 'positions': [0.0], 'support_positions': [0.1, 0.9]},
    {'load': "-13*P", 'positions': [0.5], 'support_positions': [0.1, 0.9]},
    {'load': "-13*P", 'positions': [1.0], 'support_positions': [0.1, 0.9]},
    {'load': "-13*P", 'positions': [0.0, 0.5], 'support_positions': [0.1, 0.9]},
    {'load': "-13*P", 'positions': [0.5, 1.0], 'support_positions': [0.1, 0.9]},
    {'load': "-13*P", 'positions': [0.0, 1.0], 'support_positions': [0.1, 0.9]},
]

# ============================================================================
# Generate Initial Beam Analysis Dataset
# ============================================================================
# This creates the initial dataset by generating all beam configurations
# and solving them symbolically. The dataset includes reactions, moments,
# deflections, and other beam analysis data.

# Then use create_huggingface_dataset instead of create_huggingface_dataset
dataset = create_huggingface_dataset(
    lengths=lengths,
    load_configs=load_configs,  # Use load_configs instead of loads/n_positions
    q_val=q_val,
    M_val=M_val,
    E_val=E_val,
    I_val=I_val
)

# ============================================================================
# Save Intermediary Dataset
# ============================================================================

# Create intermediary directory if it doesn't exist
os.makedirs(intermediary_dir, exist_ok=True)

# Save the initial dataset to the intermediary directory
# This preserves the state before QA generation
intermediary_filename = os.path.join(intermediary_dir, "01_initial_dataset.json")
print(f"\nSaving intermediary dataset to {intermediary_filename}...")
dataset.to_json(intermediary_filename)
print(f"Saved intermediary dataset: {intermediary_filename}")

Generating beam dataset...
Generating 24 beam configurations with explicit load positions...
Processing config 1/24: L=9*L, Load=-13*P, Positions=[0.3333]

                                    Beam points                                    
     Coordinate              Type                 Load                Moment       
-----------------------------------------------------------------------------------
         0              Pinned Support             0                    0          
      2.9997*L         Continuity point          -13*P                  0          
        9*L                 Roller                 0                    0          


                                   Beam segments                                   
        Span            Young modulus           Inertia          Distributed load  
-----------------------------------------------------------------------------------
[   0   - 2.9997*L ]          E                    I                    0          
[ 

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Saved intermediary dataset: intermediary_datasets/01_initial_dataset.json


### Syntactic prompt generation

This section generates natural language prompts from beam configuration data for question answering.

In [9]:
# ============================================================================
# FIX: Updated prompt_text_from_load_and_params_custom to handle None and multiple loads
# ============================================================================
# The old function doesn't handle None values. Use this fixed version or use _v2 instead.

def prompt_text_from_load_and_params_custom(sample):
    """
    Generate prompt text using only load_position and parameters columns.
    Fixed version that handles None values and multiple loads.
    
    Args:
        sample: Dictionary containing the sample data with load_position and parameters columns
        
    Returns:
        str: The prompt text combining load position and parameters information
    """
    # Check for new format (multiple loads) first
    load_positions = sample.get("load_positions")
    load_values = sample.get("load_values")
    load_position = sample.get("load_position")  # Legacy format
    parameters_str = sample.get("parameters")
    support_positions = sample.get("support_positions")
    
    # Parse parameters if it's a JSON string
    parameters = json.loads(parameters_str)
    
    # Extract individual parameters
    P = str(parameters.get("P", ""))
    q = str(parameters.get("q", ""))
    L = str(parameters.get("L", ""))
    M = str(parameters.get("M", ""))
    E = str(parameters.get("E", ""))
    I = str(parameters.get("I", ""))

    prompt_parts = []

    prompt_parts.append(f"The beam has a length of {L}.")
    prompt_parts.append(f"The beam has a Young's modulus of {E}.")
    prompt_parts.append(f"The beam has a moment of inertia of {I}.")

    L_unitless = float(L.split("*L")[0])
    
    # Handle multiple loads (new format)
    if load_positions is not None and load_values is not None:
        # Add information for each load
        load_descriptions = []
        for pos, load_val in zip(load_positions, load_values):
            load_descriptions.append(f"point load of {load_val} at x={round(pos*L_unitless, 3)}*L")
        
        if len(load_descriptions) == 1:
            prompt_parts.append(f"There is an applied {load_descriptions[0]}.")
        else:
            prompt_parts.append(f"There are applied loads: {', '.join(load_descriptions)}.")
        
        # Check if any load is negative (downward)
        if any("-" in str(lv) for lv in load_values):
            prompt_parts.append("A negative load means the load is applied downward.")
    
    # Handle single load (legacy format) - only if load_position is not None
    elif load_position is not None:
        # Calculate load location from load_position
        if L != "L":
            L_unitless = float(L.replace("*L", ""))  # Strip "*L" from string and convert to float
            location_of_load = float(load_position) * L_unitless
        else:
            location_of_load = float(load_position)

        # Add point load information
        if P and P != "0":
            prompt_parts.append(f"There is an applied point load of {P} at x={location_of_load}*L.")
            if "-" in P:
                prompt_parts.append("A negative load means the load is applied downward.")
    
    # Add moment information if present
    if M and M != "0":
        prompt_parts.append(f"There is an applied moment of {M}.")
    
    # Add distributed load information if present
    if q and q != "0":
        prompt_parts.append(f"There is a distributed load of {q}.")

    pinned_support_pos = support_positions[0]
    roller_support_pos = support_positions[1]
    
    # Add support information (default: pin at 0, roller at L)
    prompt_parts.append(f"The beam has a pin support at x={round(pinned_support_pos*L_unitless, 3)}*L and a roller support at x={round(roller_support_pos*L_unitless, 3)}*L.")
    
    return "\n".join(prompt_parts)

In [10]:
for idx, sample in enumerate(dataset.select(range(24))):
    print(f"\n--- Sample {idx+1} ---")
    prompt = prompt_text_from_load_and_params_custom(sample)
    print(prompt)


--- Sample 1 ---
The beam has a length of 9*L.
The beam has a Young's modulus of E.
The beam has a moment of inertia of I.
There is an applied point load of -13*P at x=3.0*L.
A negative load means the load is applied downward.
The beam has a pin support at x=0.0*L and a roller support at x=9.0*L.

--- Sample 2 ---
The beam has a length of 9*L.
The beam has a Young's modulus of E.
The beam has a moment of inertia of I.
There is an applied point load of -13*P at x=6.0*L.
A negative load means the load is applied downward.
The beam has a pin support at x=0.0*L and a roller support at x=9.0*L.

--- Sample 3 ---
The beam has a length of 9*L.
The beam has a Young's modulus of E.
The beam has a moment of inertia of I.
There is an applied point load of -13*P at x=1.125*L.
A negative load means the load is applied downward.
The beam has a pin support at x=0.0*L and a roller support at x=9.0*L.

--- Sample 4 ---
The beam has a length of 9*L.
The beam has a Young's modulus of E.
The beam has a m

### LLM-Based Question Generation

This section uses a language model to generate natural language questions from the beam configurations. The LLM takes beam parameters as input and generates questions asking about reaction forces at the supports.

In [11]:
# Import LLM inference library (vLLM for efficient GPU inference)
from vllm import LLM, SamplingParams

# Import custom prompt generation functions
from beam_prompt_calculator import *

INFO 11-18 15:15:05 __init__.py:190] Automatically detected platform cuda.


In [12]:
# ============================================================================
# LLM Configuration for QA Generation
# ============================================================================
# Configure the language model for question generation.
# The model generates multiple question variations per beam configuration.

LLM_CONFIG = {
    "model_name": "RedHatAI/DeepSeek-R1-Distill-Qwen-7B-quantized.w8a8",  # Quantized model for efficiency
    "max_length": 5120,  # Maximum sequence length
    "temperature": 0.6,  # Controls randomness (lower = more deterministic)
    "top_p": 0.9,  # Nucleus sampling parameter
    "n_outputs": 4  # Number of question variations to generate per sample
}

In [13]:
# ============================================================================
# System Prompt for Question Generation
# ============================================================================
# This prompt instructs the LLM on how to generate questions about beam
# reaction forces. The model uses chain-of-thought reasoning (with redacted
# reasoning tags) to generate high-quality questions.

PROMPT_Q_SYSTEM = """
You are a question generation assistant. You will be given information about the setup of a statically loaded beam. Your task is to generate a question that asks the reader to calculate the reaction forces at the supports.

Generate a single, self-contained question that includes all the provided details from the setup below. All details are correct. The question should be short and concise. Use limited time reasoning.

The question needs to state the length of the beam, the location and type of the supports of the beam, and the location, magnitude, direction and type of the loads applied to the beam.

Following the think tag concluding the reasoning section, return only the question and no other dialogue referencing the prompt or the setup.
"""

In [14]:
class LLMManager:
    """Manages LLM model loading and generation."""
    
    def __init__(self, config):
        self.config = config
        self.llm = None
        self.sampling_params = None
    
    def load_model(self):
        """Load the LLM model using vLLM."""
        print(f"Loading LLM model: {self.config['model_name']}...")
        self.llm = LLM(
            model=self.config['model_name'],
            max_model_len=self.config['max_length'],
            dtype="half"  # Use float16 for GPU compatibility (works on Quadro RTX 5000 and L4 GPUs)
        )
        self.sampling_params = SamplingParams(
            temperature=self.config['temperature'],
            max_tokens=self.config['max_length'],
            top_p=self.config['top_p'],
            n=self.config['n_outputs']
        )
        print("Model loaded successfully!")
    
    def generate_responses(self, prompts, system_prompt):
        """Generate responses for a batch of prompts."""
        # Add <think> if not already present
        full_prompts = []
        for prompt in prompts:
            if "<think>" not in prompt:
                full_prompt = system_prompt + "\n\n" + prompt + "\n<think>\n"
            else:
                full_prompt = system_prompt + "\n\n" + prompt + "\n"
            full_prompts.append(full_prompt)
        
        outputs = self.llm.generate(full_prompts, self.sampling_params)
        
        results = []
        for output in outputs:
            prompt_responses = []
            for single_output in output.outputs:
                generated_text = single_output.text.strip()
                prompt_responses.append(generated_text if generated_text else "No response generated")
            results.append(prompt_responses)
        
        return results

In [15]:
# ============================================================================
# UPDATED FUNCTION: generate_qa_for_dataset with multiple load support
# ============================================================================

def generate_qa_for_dataset(dataset, llm_manager, use_v2_prompt=True):
    """
    Generate questions for a dataset with support for multiple loads.
    
    Args:
        dataset: HuggingFace Dataset object
        llm_manager: LLMManager instance (must have model loaded)
        use_v2_prompt: If True, use the new prompt function that handles multiple loads
    
    Returns:
        Dataset with added 'llm_response_Q' and 'prompt_Q' columns
    """
    # Convert to list for easier processing
    dataset_list = list(dataset)
    
    print(f"Generating QA for {len(dataset_list)} samples...")
    
    # Prepare prompts for question generation (all at once)
    prompts_Q = []
    
    for sample in dataset_list:
        # Choose prompt function based on flag
        if use_v2_prompt:
            prompt_Q_input = prompt_text_from_load_and_params_custom(sample)
        else:
            prompt_Q_input = prompt_text_from_load_and_params_custom(sample)
        prompts_Q.append(prompt_Q_input)
    
    # Generate questions for all samples at once
    print("Generating questions...")
    responses_Q = llm_manager.generate_responses(prompts_Q, PROMPT_Q_SYSTEM)
    
    # Add responses to samples
    processed_data = []
    for j, sample in enumerate(dataset_list):
        sample["prompt_Q"] = prompts_Q[j]
        sample["llm_response_Q"] = responses_Q[j]  # List of n_outputs questions (cleaned)
        processed_data.append(sample)
    
    # Create new dataset with QA data
    enhanced_dataset = Dataset.from_list(processed_data)
    print(f"QA generation completed! Dataset now has {len(enhanced_dataset)} samples.")
    
    return enhanced_dataset


In [16]:
# Initialize LLM manager
llm_manager = LLMManager(LLM_CONFIG)

# Load the model (this may take a few minutes)
llm_manager.load_model()

# Generate questions and reasoning traces (all samples at once)
dataset_with_qa = generate_qa_for_dataset(
    dataset=dataset,
    llm_manager=llm_manager
)

# Update your dataset variable
dataset = dataset_with_qa

# Save QA-enhanced intermediary dataset
qa_filename = os.path.join(intermediary_dir, "02_dataset_with_qa.json")
print(f"\nSaving QA-enhanced dataset to {qa_filename}...")
dataset.to_json(qa_filename)
print(f"Saved intermediary dataset: {qa_filename}")

Loading LLM model: RedHatAI/DeepSeek-R1-Distill-Qwen-7B-quantized.w8a8...
WARNING 11-18 15:15:06 config.py:2386] Casting torch.bfloat16 to torch.float16.
INFO 11-18 15:15:12 config.py:542] This model supports multiple tasks: {'reward', 'classify', 'embed', 'generate', 'score'}. Defaulting to 'generate'.
INFO 11-18 15:15:14 llm_engine.py:234] Initializing a V0 LLM engine (v0.7.2) with config: model='RedHatAI/DeepSeek-R1-Distill-Qwen-7B-quantized.w8a8', speculative_config=None, tokenizer='RedHatAI/DeepSeek-R1-Distill-Qwen-7B-quantized.w8a8', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=5120, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=compressed-tensors, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='x

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


INFO 11-18 15:15:17 model_runner.py:1115] Loading model weights took 8.1650 GB
INFO 11-18 15:15:19 worker.py:267] Memory profiling takes 1.39 seconds
INFO 11-18 15:15:19 worker.py:267] the current vLLM instance can use total_gpu_memory (15.55GiB) x gpu_memory_utilization (0.90) = 14.00GiB
INFO 11-18 15:15:19 worker.py:267] model weights take 8.17GiB; non_torch_memory takes 0.03GiB; PyTorch activation peak memory takes 1.42GiB; the rest of the memory reserved for KV Cache is 4.38GiB.
INFO 11-18 15:15:19 executor_base.py:110] # CUDA blocks: 5120, # CPU blocks: 4681
INFO 11-18 15:15:19 executor_base.py:115] Maximum concurrency for 5120 tokens per request: 16.00x
INFO 11-18 15:15:21 model_runner.py:1434] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error occurs during cudagraph capture, consider decreasing `gpu_memory_util

Capturing CUDA graph shapes: 100%|██████████| 35/35 [00:22<00:00,  1.53it/s]

INFO 11-18 15:15:44 model_runner.py:1562] Graph capturing finished in 23 secs, took 0.22 GiB
INFO 11-18 15:15:44 llm_engine.py:431] init engine (profile, create kv cache, warmup model) took 26.95 seconds


Model loaded successfully!
Generating QA for 24 samples...
Generating questions...


Processed prompts:  25%|██▌       | 24/96 [00:59<02:57,  2.46s/it, est. speed input: 98.43 toks/s, output: 671.67 toks/s] 

QA generation completed! Dataset now has 24 samples.

Saving QA-enhanced dataset to intermediary_datasets/02_dataset_with_qa.json...


Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Saved intermediary dataset: intermediary_datasets/02_dataset_with_qa.json


### Post-Processing: Extract Answers and Clean Dataset

This section extracts ground-truth answers from the solved beam equations and cleans up the dataset by:
- Extracting questions from LLM responses (removing reasoning traces)
- Extracting reaction force answers from the beam solutions
- Renaming columns for consistency
- Saving the final processed dataset

In [17]:
# ============================================================================
# Post-Processing Functions
# ============================================================================

def extract_question_from_response(response_text):
    """
    Extract the question from an LLM response that may contain reasoning traces.
    
    LLM responses often include reasoning traces wrapped in tags. This function
    extracts only the final question text after the reasoning section.
    
    Args:
        response_text: String containing LLM response with potential reasoning traces
    
    Returns:
        str: Clean question text, or empty string if not found
    """
    # Look for the closing tag that marks the end of reasoning
    closing_tag = '</think>'
    if closing_tag in response_text:
        # Split on the closing tag and take everything after it
        parts = response_text.split(closing_tag)
        if len(parts) > 1:
            question = parts[-1].strip()
            if question:
                return question
    return ""

def extract_cleaned_responses_Q_from_row(row):
    """
    Extract cleaned questions from a dataset row.
    
    Processes the llm_response_Q column which contains a list of raw LLM responses
    and extracts clean questions from each.
    
    Args:
        row: Dataset row containing 'llm_response_Q' field
    
    Returns:
        list: List of cleaned question strings
    """
    llm_resp_list = row["llm_response_Q"]
    if isinstance(llm_resp_list, list):
        return [extract_question_from_response(resp) for resp in llm_resp_list]
    return []

def extract_cleaned_responses_Q_from_row(row):
    llm_resp_list = row["llm_response_Q"]
    if isinstance(llm_resp_list, list):
        return [extract_question_from_response(resp) for resp in llm_resp_list]
    return []

def extract_preferred_answer(reactions_data):
    """
    Extract the reactions array from the reactions column data and format as simple values.
    Expected format: {"header": "Exterior Reactions", "reactions": [...]}
    Returns: List of strings where values are the "value" field from each reaction
    """
    try:
        if isinstance(reactions_data, str):
            reactions_dict = json.loads(reactions_data)
        else:
            reactions_dict = reactions_data
            
        if isinstance(reactions_dict, dict) and "reactions" in reactions_dict:
            reactions = reactions_dict["reactions"]
            if isinstance(reactions, list):
                # Extract just the "value" field from each reaction and remove asterisks
                values = [reaction.get("value", "").replace("*", "") for reaction in reactions]
                return values
        return None
    except (json.JSONDecodeError, TypeError, KeyError):
        return None

def add_answer_column(example):
    """Add answer column by extracting preferred answers from reactions"""
    answer = extract_preferred_answer(example.get("reactions"))
    example["answer"] = answer if answer else []
    return example

In [18]:
# ============================================================================
# Clean Questions and Extract Answers
# ============================================================================

# Extract cleaned questions from LLM responses (remove reasoning traces)
print("Extracting cleaned questions from LLM responses...")
cleaned_responses_Q = [extract_cleaned_responses_Q_from_row(row) for row in dataset]

# Add cleaned questions as 'query' column and remove intermediate columns
dataset = dataset.add_column("query", cleaned_responses_Q)
dataset = dataset.remove_columns(["llm_response_Q", "prompt_Q"])
print("Renamed column 'llm_response_Q' to 'query' and removed 'prompt_Q'.")

# Extract ground-truth answers from the reactions field
print("\nExtracting answers from reactions column...")
dataset = dataset.map(add_answer_column)
print(f"Added 'answer' column. Sample: {dataset[0]['answer']}")

# ============================================================================
# Save Final Processed Dataset
# ============================================================================

# Save the final dataset locally before uploading to Hub
final_filename = os.path.join(intermediary_dir, "03_final_dataset.json")
print(f"\nSaving final dataset to {final_filename}...")
dataset.to_json(final_filename)
print(f"Saved final dataset: {final_filename}")


Extracting cleaned questions from LLM responses...
Renamed column 'llm_response_Q' to 'query' and removed 'prompt_Q'.

Extracting answers from reactions column...


Map:   0%|          | 0/24 [00:00<?, ? examples/s]

Added 'answer' column. Sample: ['8.6671P', '4.3329P']

Saving final dataset to intermediary_datasets/03_final_dataset.json...


Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Saved final dataset: intermediary_datasets/03_final_dataset.json


In [19]:
# ============================================================================
# Upload Final Dataset to HuggingFace Hub
# ============================================================================
# Upload the final processed dataset to HuggingFace Hub for public access.
# The dataset includes beam configurations, symbolic solutions, LLM-generated
# questions, and ground-truth answers.

print(f"\nUploading final dataset to HuggingFace Hub: {repo_name}...")
dataset.push_to_hub(repo_name, token=hf_token, private=private)
print(f"Dataset successfully uploaded to https://huggingface.co/datasets/{repo_name}")



Uploading final dataset to HuggingFace Hub: tphage/BeamRL-EvalData...


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


README.md: 0.00B [00:00, ?B/s]

Dataset successfully uploaded to https://huggingface.co/datasets/tphage/BeamRL-EvalData


In [20]:
first_row = dataset[0]
last_row = dataset[len(dataset)-1]
all_columns = list(dataset.features)
last_two_cols = all_columns[-2:]

print("First row (last two columns):")
for col in last_two_cols:
    print(f"{col}:")
    for elem in first_row[col]:
        print(elem)
print("Last row (last two columns):")
for col in last_two_cols:
    print(f"{col}:")
    for elem in last_row[col]:
        print(elem)

First row (last two columns):
query:
Question:  
Determine the reaction forces at the pin support (x=0.0*L) and the roller support (x=9.0*L) for a statically loaded beam with a length of 9*L, a point load of -13*P at x=3.0*L, and supports at x=0.0*L (pin) and x=9.0*L (roller).
Given a beam with a length of 9*L, supported by a pin at x=0.0*L and a roller at x=9.0*L, and subjected to a downward point load of -13*P at x=3.0*L, calculate the reaction forces at the supports.  

Question:  
Calculate the reaction forces at the pin support (R_A) and the roller support (R_B) for a beam with a length of 9*L, supported at x=0.0*L (pin) and x=9.0*L (roller), subjected to a downward point load of -13*P at x=3.0*L.
Question:  
Determine the reaction forces at the pin support (R_A) and the roller support (R_B) for a beam with a length of 9*L, supported at x=0.0*L (pin) and x=9.0*L (roller), with a downward point load of -13*P applied at x=3.0*L.
Given a beam with a length of 9*L, supported by a pin 